# Тренувальний ноутбук: лінійна регресія та регуляризація

**Курс:** Машинне навчання  
**Тема:** Linear Regression, Ridge, Lasso, Elastic Net

Цей ноутбук відповідає лекції про лінійну регресію та побудований як практичне продовження теоретичного матеріалу. У прикладах послідовно розглядаються проста й множинна лінійна регресія, метрики, залишки, недонавчання і перенавчання, L1/L2-регуляризація, Ridge, Lasso, Elastic Net, масштабування, крос-валідація, `GridSearchCV`, `Pipeline`, `ColumnTransformer` та навчальні криві.

У ноутбуці є два типи комірок:

- **Приклад** — готовий код, який потрібно виконати, проаналізувати та пояснити.
- **Самостійне завдання** — окрема кодова комірка з позначкою `TODO`. Її потрібно заповнити власноруч. Готового розв'язку в ноутбуці немає.

Рекомендація: виконувати комірки послідовно зверху вниз і після кожного блоку формулювати короткий висновок своїми словами.

## 0. Імпорт бібліотек і налаштування середовища

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)
from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_validate,
    GridSearchCV,
    learning_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Середовище готове до роботи.')

## 1. Проста лінійна регресія

Для однієї ознаки модель має вигляд

$$
\hat y = b_0 + b_1x.
$$

Тут $b_0$ — вільний член, а $b_1$ показує, на скільки одиниць у середньому змінюється прогноз $\hat y$ при збільшенні $x$ на одну одиницю.

Нижче згенеруємо дані, у яких істинна залежність приблизно дорівнює $y=5+2.4x$, але спостереження містять випадковий шум.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

X_simple = np.linspace(0, 10, 60).reshape(-1, 1)
y_simple = 5 + 2.4 * X_simple.ravel() + rng.normal(0, 2.5, size=len(X_simple))

simple_model = LinearRegression()
simple_model.fit(X_simple, y_simple)
y_simple_pred = simple_model.predict(X_simple)

print(f'Вільний член b0: {simple_model.intercept_:.3f}')
print(f'Коефіцієнт b1: {simple_model.coef_[0]:.3f}')

plt.figure(figsize=(8, 5))
plt.scatter(X_simple.ravel(), y_simple, alpha=0.75, label='Спостереження')
plt.plot(X_simple.ravel(), y_simple_pred, linewidth=2.5, label='Лінія регресії')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Проста лінійна регресія')
plt.legend()
plt.show()

**Що потрібно побачити.** Коефіцієнт нахилу має бути близьким до 2.4, але не обов'язково дорівнювати йому точно, оскільки дані містять шум. `fit()` оцінює параметри за навчальними даними, а `predict()` лише обчислює прогноз для заданих значень ознак.

### Самостійне завдання 1

1. Створіть новий масив `X_new` зі значеннями 2, 5 і 8.
2. Отримайте прогнози моделі.
3. Виведіть таблицю з колонками `x` та `prediction`.
4. Одним реченням поясніть зміст отриманого коефіцієнта `b1`.

In [ ]:
# нові значення x (модель чекає 2D масив)
X_new = np.array([2, 5, 8]).reshape(-1, 1)
predictions = simple_model.predict(X_new)

pred_table = pd.DataFrame({'x': X_new.ravel(), 'prediction': predictions})
pred_table

**Пояснення:** b1 = 2.451, тобто якщо x збільшити на 1, то прогноз y в середньому зростає приблизно на 2.45 (близько до справжнього нахилу 2.4, різниця через шум).

## 2. Метод найменших квадратів і ручний розрахунок коефіцієнтів

Для простої регресії коефіцієнт нахилу можна обчислити без готової моделі:

$$
b_1 = \frac{\sum (x_i-\bar x)(y_i-\bar y)}{\sum (x_i-\bar x)^2},
\qquad
b_0 = \bar y-b_1\bar x.
$$

Чисельник у формулі для $b_1$ відображає спільну зміну $x$ і $y$, а знаменник — варіативність самої ознаки $x$. Після знаходження нахилу вільний член підбирається так, щоб пряма проходила через точку $(\bar x, \bar y)$.

In [ ]:
x = np.array([1, 2, 3, 4, 5], dtype=float)
y = np.array([3.0, 5.2, 6.8, 9.1, 10.9])

x_bar = x.mean()
y_bar = y.mean()

b1_manual = ((x - x_bar) * (y - y_bar)).sum() / ((x - x_bar) ** 2).sum()
b0_manual = y_bar - b1_manual * x_bar

model_check = LinearRegression().fit(x.reshape(-1, 1), y)

print(f'Ручний розрахунок: b0={b0_manual:.4f}, b1={b1_manual:.4f}')
print(f'scikit-learn:     b0={model_check.intercept_:.4f}, b1={model_check.coef_[0]:.4f}')

### Самостійне завдання 2

Використайте масиви `x2 = [2, 4, 6, 8, 10]` та `y2 = [7, 11, 16, 19, 25]`. Власноруч обчисліть `b0` і `b1` за формулами, а потім перевірте результат за допомогою `LinearRegression`.

In [ ]:
x2 = np.array([2, 4, 6, 8, 10], dtype=float)
y2 = np.array([7, 11, 16, 19, 25], dtype=float)

# середні
x2_bar = x2.mean()
y2_bar = y2.mean()

# коефіцієнти за формулами МНК
b1_2 = ((x2 - x2_bar) * (y2 - y2_bar)).sum() / ((x2 - x2_bar) ** 2).sum()
b0_2 = y2_bar - b1_2 * x2_bar

# перевірка через sklearn
check2 = LinearRegression().fit(x2.reshape(-1, 1), y2)

print(f'Середні: x={x2_bar}, y={y2_bar}')
print(f'Ручний розрахунок: b0={b0_2:.4f}, b1={b1_2:.4f}')
print(f'scikit-learn:      b0={check2.intercept_:.4f}, b1={check2.coef_[0]:.4f}')

**Висновок:** ручний розрахунок і LinearRegression дали однакові коефіцієнти: b0 = 2.4, b1 = 2.2. Отже рівняння регресії y = 2.4 + 2.2x.

## 3. Множинна лінійна регресія

У випадку кількох ознак модель має вигляд

$$
\hat y = b_0+b_1x_1+b_2x_2+\dots+b_px_p.
$$

Коефіцієнт $b_j$ інтерпретується як зміна прогнозу при збільшенні ознаки $x_j$ на одну одиницю **за інших рівних умов**, тобто при фіксованих значеннях інших ознак.

Для практики використаємо вбудований набір `diabetes`, тому ноутбук не потребує завантаження даних з Інтернету.

In [ ]:
diabetes = load_diabetes(as_frame=True)
X = diabetes.data.copy()
y = diabetes.target.copy()

print('Розмір X:', X.shape)
print('Розмір y:', y.shape)
display(X.head())

In [ ]:
y

In [ ]:
multi_model = LinearRegression()
multi_model.fit(X, y)

coef_table = pd.DataFrame({
    'feature': X.columns,
    'coefficient': multi_model.coef_
}).sort_values('coefficient', key=np.abs, ascending=False)

display(coef_table)
print(f'Intercept: {multi_model.intercept_:.3f}')

Зверніть увагу: великий за модулем коефіцієнт не завжди означає, що ознака є «найважливішою». Порівняння коефіцієнтів залежить від масштабу ознак та їх взаємної кореляції.

## 4. Train/test та наївний baseline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

linear = LinearRegression()
linear.fit(X_train, y_train)
linear_pred = linear.predict(X_test)



`DummyRegressor` не використовує закономірності в ознаках і в цьому прикладі прогнозує середнє значення цільової змінної з train. Якщо реальна модель не перевершує такий baseline, необхідно переглянути ознаки, постановку задачі або сам алгоритм.

## 5. Метрики якості регресії

У лекції розглядалися MAE, MSE, RMSE та $R^2$. Вони відповідають на різні питання.

- **MAE** показує середню абсолютну величину помилки.
- **MSE** сильніше штрафує великі помилки через піднесення залишку до квадрата.
- **RMSE** є коренем з MSE і вимірюється в тих самих одиницях, що й цільова змінна.
- **$R^2$** порівнює модель із прогнозом середнім значенням цільової змінної. На нових даних він може бути від'ємним.

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'MSE': mean_squared_error(y_true, y_pred),
        'RMSE': root_mean_squared_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
        'MAPE': (np.abs(y_true - y_pred) / y_true).mean() * 100
    }

metrics_baseline = regression_metrics(y_test, baseline_pred)
metrics_linear = regression_metrics(y_test, linear_pred)

pd.DataFrame([metrics_baseline, metrics_linear], index=['DummyRegressor', 'LinearRegression'])

### Самостійне завдання 3

Обчисліть MAE, RMSE і $R^2$ окремо, **не використовуючи** функцію `regression_metrics`. Порівняйте `DummyRegressor` і `LinearRegression`. У Markdown-комірці нижче поясніть, яка модель краща і чому не варто робити висновок лише за однією метрикою.

In [ ]:
# рахуємо метрики за формулами, без regression_metrics
rows3 = []
for name, pred in [('DummyRegressor', baseline_pred), ('LinearRegression', linear_pred)]:
    err = y_test - pred
    mae = np.abs(err).mean()
    rmse = np.sqrt((err ** 2).mean())
    r2 = 1 - (err ** 2).sum() / ((y_test - y_test.mean()) ** 2).sum()
    rows3.append({'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})

pd.DataFrame(rows3).set_index('model')

**Ваш висновок:**

LinearRegression краща за DummyRegressor за всіма трьома метриками: MAE 42.8 проти 64.0, RMSE 53.9 проти 73.2, R2 0.45 проти -0.01. У DummyRegressor R2 навіть трохи менше нуля, бо він прогнозує середнє з train, а на test середнє трохи інше. Робити висновок лише за однією метрикою не варто, бо вони показують різне: MAE це середня помилка, RMSE сильніше штрафує великі помилки, а R2 порівнює модель з прогнозом середнім. Наприклад, R2 = 0.45 здається не дуже високим, але MAE порівняно з baseline зменшилась приблизно на третину.

## 6. Залишки та діагностика моделі

Залишок для об'єкта визначається як

$$
e_i = y_i-\hat y_i.
$$

Якщо модель добре описує систематичну частину залежності, на графіку `prediction → residual` не повинно бути вираженої структури. Дуга може вказувати на нелінійність, а «воронка» — на зміну дисперсії помилки.

In [ ]:
residuals = y_test - linear_pred

plt.figure(figsize=(8, 5))
plt.scatter(linear_pred, residuals, alpha=0.75)
plt.axhline(0, linestyle='--', linewidth=2)
plt.xlabel('Прогноз')
plt.ylabel('Залишок y - ŷ')
plt.title('Графік залишків для LinearRegression')
plt.show()

## 7. Недонавчання і перенавчання на поліноміальній регресії

Лінійна модель може недонавчатися, якщо реальна залежність нелінійна. Один зі способів збільшити гнучкість — створити поліноміальні ознаки. Але занадто високий ступінь полінома збільшує ризик перенавчання.

Порівняємо ступені 1, 4 і 15 на невеликій синтетичній вибірці.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
X_poly = np.sort(rng.uniform(-3, 3, 45)).reshape(-1, 1)
y_true_poly = np.sin(X_poly.ravel()) + 0.25 * X_poly.ravel()
y_poly = y_true_poly + rng.normal(0, 0.22, size=len(X_poly))

X_p_train, X_p_test, y_p_train, y_p_test = train_test_split(
    X_poly, y_poly, test_size=0.35, random_state=RANDOM_STATE
)

def fit_polynomial(degree):
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('reg', LinearRegression()),
    ])
    model.fit(X_p_train, y_p_train)
    train_pred = model.predict(X_p_train)
    test_pred = model.predict(X_p_test)
    return model, {
        'degree': degree,
        'train_rmse': root_mean_squared_error(y_p_train, train_pred),
        'test_rmse': root_mean_squared_error(y_p_test, test_pred),
    }

results = []
models_poly = {}
for degree in [1, 4, 15]:
    m, res = fit_polynomial(degree)
    models_poly[degree] = m
    results.append(res)

pd.DataFrame(results)

In [ ]:
grid = np.linspace(-3, 3, 400).reshape(-1, 1)

plt.figure(figsize=(10, 6))
plt.scatter(X_p_train, y_p_train, label='Train', alpha=0.8)
plt.scatter(X_p_test, y_p_test, label='Test', alpha=0.8)

for degree in [1, 4, 15]:
    plt.plot(grid, models_poly[degree].predict(grid), linewidth=2, label=f'degree={degree}')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Недонавчання, адекватна складність і перенавчання')
plt.legend()
plt.show()

### Самостійне завдання 4

1. Додайте до порівняння ступені 2, 6 і 10.
2. Побудуйте таблицю `degree`, `train_rmse`, `test_rmse`.
3. Визначте ступінь із найменшим `test_rmse`.
4. Поясніть, чому мінімальна train-помилка не гарантує найкращої моделі.

In [ ]:
results4 = []
for degree in [1, 2, 4, 6, 10, 15]:
    _, res = fit_polynomial(degree)
    results4.append(res)

results4 = pd.DataFrame(results4)
display(results4)

# ступінь з найменшою помилкою на test
best4 = results4.loc[results4['test_rmse'].idxmin()]
print('Найменший test_rmse:', int(best4['degree']), 'ступінь,', round(best4['test_rmse'], 4))

**Висновок:** найменший test_rmse у ступеня 10 (0.192), трохи гірше ступені 6 і 4 (0.196 і 0.203). Ступені 1 і 2 недонавчаються, у них велика помилка і на train, і на test (близько 0.39 і 0.46). У ступеня 15 найменша помилка на train (0.091), але на test вона більша (0.244), ніж у ступенів 4, 6 і 10, тобто модель перенавчилась. Мінімальна train-помилка не гарантує найкращої моделі, бо складна модель підлаштовується під шум у навчальних точках і гірше працює на нових даних.

## 8. Регуляризація: загальна ідея

Коли модель має багато ознак або ознаки сильно корельовані, коефіцієнти можуть ставати нестабільними. Регуляризація додає до функції втрат штраф за великі коефіцієнти.

Для Ridge використовується L2-штраф

$$
\text{MSE}+\alpha\sum_{j=1}^{p} b_j^2,
$$

для Lasso — L1-штраф

$$
\text{MSE}+\alpha\sum_{j=1}^{p}|b_j|.
$$

Параметр $\alpha$ керує силою регуляризації. Якщо $\alpha=0$, штраф зникає. Надто велике $\alpha$ може надмірно стиснути коефіцієнти й спричинити недонавчання.

## 9. Чому перед Ridge, Lasso та Elastic Net потрібне масштабування

In [ ]:
# Штучно змінюємо масштаби кількох ознак, щоб побачити роль StandardScaler.
X_scaled_demo = X.copy()
X_scaled_demo['bmi'] = X_scaled_demo['bmi'] * 1000
X_scaled_demo['bp'] = X_scaled_demo['bp'] * 0.01

Xsd_train, Xsd_test, ysd_train, ysd_test = train_test_split(
    X_scaled_demo, y, test_size=0.2, random_state=RANDOM_STATE
)

ridge_no_scaling = Ridge(alpha=10.0).fit(Xsd_train, ysd_train)
ridge_pipeline = Pipeline([
    ('scale', StandardScaler()),
    ('reg', Ridge(alpha=10.0)),
]).fit(Xsd_train, ysd_train)

print('Ridge без масштабування RMSE:', root_mean_squared_error(ysd_test, ridge_no_scaling.predict(Xsd_test)))
print('Ridge + StandardScaler RMSE:', root_mean_squared_error(ysd_test, ridge_pipeline.predict(Xsd_test)))

Штраф застосовується безпосередньо до коефіцієнтів. Якщо ознаки вимірюються в дуже різних масштабах, одна й та сама величина коефіцієнта не має однакового змісту. Тому `StandardScaler` зазвичай включають у `Pipeline` перед регуляризованою лінійною моделлю.

## 10. Ridge-регресія: L2-регуляризація

In [ ]:
ridge = Pipeline([
    ('scale', StandardScaler()),
    ('reg', Ridge(alpha=1.0)),
])

ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)

print(regression_metrics(y_test, ridge_pred))

## 11. Lasso: L1-регуляризація

In [ ]:
lasso = Pipeline([
    ('scale', StandardScaler()),
    ('reg', Lasso(alpha=1.0, max_iter=20000)),
])

lasso.fit(X_train, y_train)
lasso_pred = lasso.predict(X_test)

lasso_coef = lasso.named_steps['reg'].coef_
print(regression_metrics(y_test, lasso_pred))
print('Кількість нульових коефіцієнтів:', np.sum(np.isclose(lasso_coef, 0)))

На відміну від Ridge, Lasso може занулювати частину коефіцієнтів. Саме тому L1-регуляризацію іноді використовують як вбудований механізм відбору ознак. Проте при сильно корельованих предикторах вибір конкретної ознаки може бути нестабільним.

## 12. Elastic Net: поєднання L1 та L2

In [ ]:
elastic = Pipeline([
    ('scale', StandardScaler()),
    ('reg', ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=20000)),
])

elastic.fit(X_train, y_train)
elastic_pred = elastic.predict(X_test)

print(regression_metrics(y_test, elastic_pred))

В Elastic Net параметр `alpha` визначає загальну силу регуляризації, а `l1_ratio` — співвідношення між L1 та L2. Значення `l1_ratio=1` наближає модель до Lasso, а `l1_ratio=0` — до Ridge.

### Самостійне завдання 5

Навчіть на одному й тому самому train/test розбитті чотири моделі: `LinearRegression`, `Ridge`, `Lasso`, `ElasticNet`. Для регуляризованих моделей використайте `StandardScaler` у `Pipeline`. Побудуйте таблицю з MAE, RMSE та $R^2$.

In [ ]:
# регуляризовані моделі зі StandardScaler, звичайна лінійна без нього
models5 = {
    'LinearRegression': LinearRegression(),
    'Ridge': Pipeline([('scale', StandardScaler()), ('reg', Ridge(alpha=1.0))]),
    'Lasso': Pipeline([('scale', StandardScaler()), ('reg', Lasso(alpha=1.0, max_iter=20000))]),
    'ElasticNet': Pipeline([('scale', StandardScaler()), ('reg', ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=20000))]),
}

rows5 = []
for name, model in models5.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows5.append({
        'model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': root_mean_squared_error(y_test, pred),
        'R2': r2_score(y_test, pred),
    })

pd.DataFrame(rows5).set_index('model')

**Висновок:** на цьому розбитті всі чотири моделі дали майже однаковий результат: R2 від 0.453 до 0.467, RMSE від 53.1 до 53.9. Найкраще вийшло в Lasso (RMSE 53.1, R2 0.467), але різниця невелика, тому сильної переваги регуляризація тут не дає.

## 13. Крос-валідація

Одне train/test розбиття залежить від випадкового поділу. K-fold cross-validation кілька разів змінює validation-частину і дає більш стійку оцінку якості моделі. Важливо, що preprocessing повинен бути всередині `Pipeline`, щоб на кожному фолді параметри масштабування оцінювалися лише з train-частини.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

ridge_cv_model = Pipeline([
    ('scale', StandardScaler()),
    ('reg', Ridge(alpha=1.0)),
])

scores = cross_validate(
    ridge_cv_model,
    X,
    y,
    cv=cv,
    scoring={
        'mae': 'neg_mean_absolute_error',
        'rmse': 'neg_root_mean_squared_error',
        'r2': 'r2',
    },
    return_train_score=True,
)

cv_table = pd.DataFrame({
    'fold': np.arange(1, 6),
    'train_R2': scores['train_r2'],
    'val_R2': scores['test_r2'],
    'val_MAE': -scores['test_mae'],
    'val_RMSE': -scores['test_rmse'],
})

display(cv_table)
print('Середній validation RMSE:', cv_table['val_RMSE'].mean())

## 14. Підбір alpha для Ridge за GridSearchCV

In [ ]:
ridge_search_model = Pipeline([
    ('scale', StandardScaler()),
    ('reg', Ridge()),
])

param_grid_ridge = {
    'reg__alpha': np.logspace(0, 5, 13)
}

ridge_search = GridSearchCV(
    ridge_search_model,
    param_grid=param_grid_ridge,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)

ridge_search.fit(X_train, y_train)

print('Найкраще alpha:', ridge_search.best_params_['reg__alpha'])
print('CV RMSE:', -ridge_search.best_score_)
print('Test RMSE:', root_mean_squared_error(y_test, ridge_search.predict(X_test)))

Тестову вибірку не використовують для підбору `alpha`. Гіперпараметр обирається всередині train за крос-валідацією, а test використовується лише після завершення вибору моделі.

## 15. Підбір alpha і l1_ratio для Elastic Net

In [ ]:
elastic_search_model = Pipeline([
    ('scale', StandardScaler()),
    ('reg', ElasticNet(max_iter=30000)),
])

param_grid_elastic = {
    'reg__alpha': [0.001, 0.01, 0.03, 0.1, 0.3, 1.0],
    'reg__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
}

elastic_search = GridSearchCV(
    elastic_search_model,
    param_grid=param_grid_elastic,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)

elastic_search.fit(X_train, y_train)

print('Найкращі параметри:', elastic_search.best_params_)
print('CV RMSE:', -elastic_search.best_score_)
print('Test RMSE:', root_mean_squared_error(y_test, elastic_search.predict(X_test)))

### Самостійне завдання 6

Виконайте `GridSearchCV` для Lasso. Перевірте не менше 10 значень `alpha` у логарифмічній шкалі. Порівняйте найкращий Lasso з найкращим Ridge за однаковою схемою CV.

In [ ]:
lasso_search_model = Pipeline([
    ('scale', StandardScaler()),
    ('reg', Lasso(max_iter=30000)),
])

# 13 значень alpha від 0.001 до 100 (логарифмічна шкала)
param_grid_lasso = {'reg__alpha': np.logspace(-3, 2, 13)}

lasso_search = GridSearchCV(
    lasso_search_model,
    param_grid=param_grid_lasso,
    cv=cv,  # та сама схема CV, що і для Ridge
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)
lasso_search.fit(X_train, y_train)

compare6 = pd.DataFrame({
    'best_alpha': [ridge_search.best_params_['reg__alpha'], lasso_search.best_params_['reg__alpha']],
    'CV_RMSE': [-ridge_search.best_score_, -lasso_search.best_score_],
    'test_RMSE': [
        root_mean_squared_error(y_test, ridge_search.predict(X_test)),
        root_mean_squared_error(y_test, lasso_search.predict(X_test)),
    ],
}, index=['Ridge', 'Lasso'])
compare6

**Висновок:** найкраще alpha для Lasso вийшло 0.12, для Ridge 1.0. За CV RMSE Lasso трохи краща (55.35 проти 55.37), на test теж трохи краща (53.69 проти 53.78). Різниця дуже маленька, тобто на цих даних обидві моделі працюють майже однаково.

## 16. Як регуляризація змінює коефіцієнти

In [ ]:
alphas = np.logspace(-3, 3, 40)
ridge_paths = []
lasso_paths = []

X_for_path = StandardScaler().fit_transform(X)

for alpha in alphas:
    ridge_m = Ridge(alpha=alpha).fit(X_for_path, y)
    lasso_m = Lasso(alpha=alpha, max_iter=30000).fit(X_for_path, y)
    ridge_paths.append(ridge_m.coef_)
    lasso_paths.append(lasso_m.coef_)

ridge_paths = np.array(ridge_paths)
lasso_paths = np.array(lasso_paths)

plt.figure(figsize=(10, 6))
for j, name in enumerate(X.columns):
    plt.plot(alphas, ridge_paths[:, j], label=name)
plt.xscale('log')
plt.xlabel('alpha')
plt.ylabel('Коефіцієнт')
plt.title('Ridge: стискання коефіцієнтів зі зростанням alpha')
plt.legend(ncol=2, fontsize=8)
plt.show()

plt.figure(figsize=(10, 6))
for j, name in enumerate(X.columns):
    plt.plot(alphas, lasso_paths[:, j], label=name)
plt.xscale('log')
plt.xlabel('alpha')
plt.ylabel('Коефіцієнт')
plt.title('Lasso: частина коефіцієнтів стає нульовою')
plt.legend(ncol=2, fontsize=8)
plt.show()

Ці графіки демонструють ключову різницю. Ridge плавно стискає коефіцієнти до нуля, тоді як Lasso може зробити частину з них точно нульовими.

## 17. Мультиколінеарність і стабільність коефіцієнтів

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = 250
x1 = rng.normal(size=n)
x2 = x1 + rng.normal(scale=0.03, size=n)  # дуже сильна кореляція
x3 = rng.normal(size=n)
y_corr = 5 * x1 + 5 * x2 + 0.5 * x3 + rng.normal(scale=1.5, size=n)

X_corr = pd.DataFrame({'x1': x1, 'x2': x2, 'x3': x3})
print(X_corr.corr())

ols_corr = LinearRegression().fit(X_corr, y_corr)
ridge_corr = Pipeline([
    ('scale', StandardScaler()),
    ('reg', Ridge(alpha=10.0)),
]).fit(X_corr, y_corr)

coef_compare = pd.DataFrame({
    'feature': X_corr.columns,
    'OLS': ols_corr.coef_,
    'Ridge_scaled': ridge_corr.named_steps['reg'].coef_,
})
coef_compare

За наявності дуже корельованих ознак окремі коефіцієнти OLS можуть бути нестабільними: невеликі зміни даних здатні суттєво змінити їх значення. Ridge зменшує variance коефіцієнтів завдяки L2-штрафу.

## 18. Категоріальні ознаки, пропуски та ColumnTransformer

У реальній табличній задачі дані часто містять числові й категоріальні ознаки, а також пропуски. Їх потрібно обробляти всередині `Pipeline`, щоб не створювати витік інформації.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = 300

housing = pd.DataFrame({
    'area': rng.normal(75, 20, n).clip(25, 150),
    'rooms': rng.integers(1, 6, n),
    'age': rng.integers(0, 70, n).astype(float),
    'district': rng.choice(['center', 'north', 'south'], n, p=[0.3, 0.35, 0.35]),
})

# Додаємо кілька пропусків.
housing.loc[rng.choice(n, 18, replace=False), 'age'] = np.nan

price = (
    1500 * housing['area']
    + 7000 * housing['rooms']
    - 500 * housing['age'].fillna(housing['age'].median())
    + housing['district'].map({'center': 40000, 'north': 12000, 'south': 0})
    + rng.normal(0, 12000, n)
)

X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    housing, price, test_size=0.2, random_state=RANDOM_STATE
)

numeric_features = ['area', 'rooms', 'age']
categorical_features = ['district']

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features),
])

housing_model = Pipeline([
    ('prep', preprocessor),
    ('reg', Ridge(alpha=1.0)),
])

housing_model.fit(X_h_train, y_h_train)
housing_pred = housing_model.predict(X_h_test)

print('Housing RMSE:', root_mean_squared_error(y_h_test, housing_pred))
print('Housing R2:', r2_score(y_h_test, housing_pred))
print('Housing MAE:', mean_absolute_error(y_h_test, housing_pred))
print('MAPE', mean_absolute_percentage_error(y_h_test, housing_pred))

In [ ]:
housing

In [ ]:
price

### Самостійне завдання 7

Додайте до таблиці `housing` нову категоріальну ознаку `condition` зі значеннями `new`, `good`, `needs_repair`. Включіть її в `ColumnTransformer`, перенавчіть модель і перевірте, чи змінилася test-помилка.

In [ ]:
# нова ознака condition (окремий генератор, щоб не залежати від попередніх комірок)
rng7 = np.random.default_rng(7)
housing2 = housing.copy()
housing2['condition'] = rng7.choice(['new', 'good', 'needs_repair'], len(housing2))

# той самий random_state, тому рядки в train/test ті самі, що й раніше
X_h2_train, X_h2_test, y_h2_train, y_h2_test = train_test_split(
    housing2, price, test_size=0.2, random_state=RANDOM_STATE
)

categorical_features2 = ['district', 'condition']

preprocessor2 = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features2),
])

housing_model2 = Pipeline([
    ('prep', preprocessor2),
    ('reg', Ridge(alpha=1.0)),
])
housing_model2.fit(X_h2_train, y_h2_train)
housing_pred2 = housing_model2.predict(X_h2_test)

pd.DataFrame({
    'RMSE': [root_mean_squared_error(y_h_test, housing_pred), root_mean_squared_error(y_h2_test, housing_pred2)],
    'MAE': [mean_absolute_error(y_h_test, housing_pred), mean_absolute_error(y_h2_test, housing_pred2)],
    'R2': [r2_score(y_h_test, housing_pred), r2_score(y_h2_test, housing_pred2)],
}, index=['без condition', 'з condition'])

**Висновок:** після додавання condition test-помилка трохи зросла: RMSE з 10925 до 11260, MAE з 8737 до 9062, R2 зменшився з 0.907 до 0.901. Так вийшло, бо condition я заповнив випадково і ціна від нього не залежить (price порахували раніше без цієї ознаки). Тобто модель отримала зайву ознаку, яка нічого не пояснює і лише трохи заважає. Якби стан квартири реально впливав на ціну, помилка мала б зменшитись.

## 19. Навчальні криві

In [ ]:
learning_model = Pipeline([
    ('scale', StandardScaler()),
    ('reg', Ridge(alpha=ridge_search.best_params_['reg__alpha'])),
])

train_sizes, train_scores, val_scores = learning_curve(
    learning_model,
    X,
    y,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    train_sizes=np.linspace(0.2, 1.0, 6),
    n_jobs=-1,
)

train_rmse_curve = -train_scores.mean(axis=1)
val_rmse_curve = -val_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_rmse_curve, marker='o', label='Train RMSE')
plt.plot(train_sizes, val_rmse_curve, marker='o', label='Validation RMSE')
plt.xlabel('Кількість навчальних об’єктів')
plt.ylabel('RMSE')
plt.title('Навчальні криві Ridge')
plt.legend()
plt.show()

Якщо train і validation помилки високі та близькі, це ознака можливого недонавчання. Якщо train-помилка значно нижча за validation, це може свідчити про високу variance та перенавчання. Навчальні криві допомагають зрозуміти, чи може збільшення обсягу даних бути корисним.

## 20. Повний приклад: коректний вибір моделі без використання test

In [ ]:
# 1. Test відкладаємо один раз.
X_final_train, X_final_test, y_final_train, y_final_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# 2. Вибір моделі відбувається тільки всередині train за CV.
model = Pipeline([
    ('scale', StandardScaler()),
    ('reg', ElasticNet(max_iter=30000)),
])

param_grid = {
    'reg__alpha': [0.001, 0.01, 0.03, 0.1, 0.3, 1.0],
    'reg__l1_ratio': [0.1, 0.5, 0.9],
}

search = GridSearchCV(
    model,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    n_jobs=-1,
)
search.fit(X_final_train, y_final_train)

# 3. Лише після завершення підбору оцінюємо test.
final_pred = search.predict(X_final_test)

print('Найкращі параметри:', search.best_params_)
print('CV RMSE:', -search.best_score_)
print('Test metrics:', regression_metrics(y_final_test, final_pred))

# 21. Підсумкове практичне завдання для самостійної роботи

Виконайте повний експеримент на наборі `diabetes`. Цей блок потрібно зробити **самостійно**, використовуючи попередні приклади лише як орієнтир.

Вимоги до роботи:

1. Завантажити `load_diabetes(as_frame=True)` і переглянути структуру даних.
2. Відкласти 20% даних у test із `random_state=42`.
3. Побудувати `DummyRegressor` і `LinearRegression` як baseline.
4. Обчислити MAE, RMSE та $R^2$ на test.
5. Побудувати Ridge, Lasso та Elastic Net через `Pipeline` із `StandardScaler`.
6. Виконати 5-fold cross-validation.
7. Для Ridge і Lasso підібрати `alpha`; для Elastic Net — `alpha` та `l1_ratio`.
8. Порівняти моделі в одній таблиці.
9. Для найкращої моделі побудувати графік `фактичні значення → прогноз` та графік залишків.
10. Проаналізувати коефіцієнти найкращої лінійної моделі.
11. Сформулювати підсумковий висновок: яка модель обрана, за якою метрикою та чому.

Нижче залишені окремі комірки для кожного етапу.

### Крок 1. Дані

In [ ]:
diabetes_task = load_diabetes(as_frame=True)
X_task = diabetes_task.data
y_task = diabetes_task.target

print('X_task:', X_task.shape)
print('y_task:', y_task.shape)
print('Пропуски:', X_task.isna().sum().sum())
display(X_task.head())

### Крок 2. Train/test

In [ ]:
Xt_train, Xt_test, yt_train, yt_test = train_test_split(
    X_task, y_task, test_size=0.2, random_state=42
)

print('Train:', Xt_train.shape)
print('Test:', Xt_test.shape)

### Крок 3. Baseline

In [ ]:
# baseline: прогноз середнім і звичайна лінійна регресія
dummy_task = DummyRegressor(strategy='mean').fit(Xt_train, yt_train)
lin_task = LinearRegression().fit(Xt_train, yt_train)

dummy_task_pred = dummy_task.predict(Xt_test)
lin_task_pred = lin_task.predict(Xt_test)

### Крок 4. Метрики

In [ ]:
def get_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': root_mean_squared_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
    }

baseline_table = pd.DataFrame(
    [get_metrics(yt_test, dummy_task_pred), get_metrics(yt_test, lin_task_pred)],
    index=['DummyRegressor', 'LinearRegression'],
)
baseline_table

### Крок 5. Ridge, Lasso, Elastic Net

In [ ]:
# параметри поки стандартні, підбір буде в кроці 7
ridge_task = Pipeline([('scale', StandardScaler()), ('reg', Ridge())])
lasso_task = Pipeline([('scale', StandardScaler()), ('reg', Lasso(max_iter=30000))])
elastic_task = Pipeline([('scale', StandardScaler()), ('reg', ElasticNet(max_iter=30000))])

### Крок 6. Крос-валідація

In [ ]:
cv_task = KFold(n_splits=5, shuffle=True, random_state=42)

task_models = {
    'LinearRegression': LinearRegression(),
    'Ridge': ridge_task,
    'Lasso': lasso_task,
    'ElasticNet': elastic_task,
}

# CV тільки на train, test не чіпаємо
cv_rows = []
for name, model in task_models.items():
    scores = cross_validate(
        model, Xt_train, yt_train, cv=cv_task,
        scoring='neg_root_mean_squared_error', return_train_score=True,
    )
    cv_rows.append({
        'model': name,
        'train_RMSE': -scores['train_score'].mean(),
        'CV_RMSE': -scores['test_score'].mean(),
    })

pd.DataFrame(cv_rows).set_index('model')

**Висновок:** з параметрами за замовчуванням LinearRegression, Ridge і Lasso мають майже однаковий CV RMSE (близько 55.4). ElasticNet гірший (56.7), бо alpha = 1.0 для нього завелике і він занадто сильно стискає коефіцієнти. Train RMSE лише трохи менший за CV RMSE (53.3-53.7 проти 55.4), тобто сильного перенавчання немає.

### Крок 7. Підбір гіперпараметрів

In [ ]:
search_ridge = GridSearchCV(
    ridge_task, {'reg__alpha': np.logspace(-3, 3, 13)},
    cv=cv_task, scoring='neg_root_mean_squared_error', n_jobs=-1,
).fit(Xt_train, yt_train)

search_lasso = GridSearchCV(
    lasso_task, {'reg__alpha': np.logspace(-3, 2, 11)},
    cv=cv_task, scoring='neg_root_mean_squared_error', n_jobs=-1,
).fit(Xt_train, yt_train)

search_elastic = GridSearchCV(
    elastic_task,
    {'reg__alpha': [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0],
     'reg__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]},
    cv=cv_task, scoring='neg_root_mean_squared_error', n_jobs=-1,
).fit(Xt_train, yt_train)

for name, s in [('Ridge', search_ridge), ('Lasso', search_lasso), ('ElasticNet', search_elastic)]:
    print(name, s.best_params_, 'CV RMSE =', round(-s.best_score_, 3))

### Крок 8. Фінальна таблиця моделей

In [ ]:
final_models = {
    'DummyRegressor': dummy_task,
    'LinearRegression': lin_task,
    'Ridge': search_ridge,
    'Lasso': search_lasso,
    'ElasticNet': search_elastic,
}

final_rows = []
for name, model in final_models.items():
    # у GridSearchCV CV RMSE вже пораховано, для інших рахуємо так само на train
    if hasattr(model, 'best_score_'):
        cv_rmse = -model.best_score_
    else:
        cv_rmse = -cross_validate(model, Xt_train, yt_train, cv=cv_task,
                                  scoring='neg_root_mean_squared_error')['test_score'].mean()
    m = get_metrics(yt_test, model.predict(Xt_test))
    final_rows.append({
        'model': name,
        'CV RMSE': cv_rmse,
        'test MAE': m['MAE'],
        'test RMSE': m['RMSE'],
        'test R2': m['R2'],
    })

# сортуємо за CV RMSE, бо модель обираємо за CV, а не за test
final_table = pd.DataFrame(final_rows).set_index('model').sort_values('CV RMSE')
display(final_table)

best_name = final_table.index[0]
best_model = final_models[best_name]
print('Найкраща модель за CV RMSE:', best_name)

**Висновок:** за CV RMSE найкраща Lasso з alpha = 0.1 (55.35), але ElasticNet, Ridge і LinearRegression відстають зовсім трохи. На test у Lasso RMSE 53.71 і R2 0.456. Усі лінійні моделі набагато кращі за DummyRegressor (RMSE 73.2).

### Крок 9. Візуалізація найкращої моделі

In [ ]:
best_pred = best_model.predict(Xt_test)
best_resid = yt_test - best_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(yt_test, best_pred, alpha=0.7)
# лінія ідеального прогнозу
axes[0].plot([yt_test.min(), yt_test.max()], [yt_test.min(), yt_test.max()], color='red', linestyle='--')
axes[0].set_xlabel('Фактичні значення')
axes[0].set_ylabel('Прогноз')
axes[0].set_title(f'{best_name}: факт і прогноз')

axes[1].scatter(best_pred, best_resid, alpha=0.7)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Прогноз')
axes[1].set_ylabel('Залишок y - ŷ')
axes[1].set_title(f'{best_name}: залишки')

plt.tight_layout()
plt.show()

**Висновок:** точки на першому графіку розкидані навколо червоної лінії досить широко, тобто прогноз неточний. Видно, що малі значення модель трохи завищує, а великі занижує. На графіку залишків немає явної дуги чи воронки, але розкид великий, більшість залишків приблизно від -100 до 100, що відповідає R2 близько 0.46.

### Крок 10. Коефіцієнти

In [ ]:
# дістаємо саму модель з GridSearchCV і Pipeline
best_est = best_model.best_estimator_ if hasattr(best_model, 'best_estimator_') else best_model
best_reg = best_est.named_steps['reg'] if hasattr(best_est, 'named_steps') else best_est

coef_task = pd.DataFrame({
    'feature': X_task.columns,
    'coefficient': best_reg.coef_,
}).sort_values('coefficient', key=np.abs, ascending=False)

display(coef_task)
print('Нульових коефіцієнтів:', int(np.isclose(best_reg.coef_, 0).sum()))

**Висновок:** найбільший вплив мають s5, s1 і bmi, далі bp і s2. Коефіцієнти s1 і s2 мають різні знаки, бо ці ознаки (показники аналізу крові) сильно корелюють між собою, тому дивитися на кожну окремо треба обережно. Lasso з alpha = 0.1 жоден коефіцієнт не занулила, найменші за модулем у s3, age і s6. Коефіцієнти пораховані після StandardScaler, тому їх можна порівнювати між собою.

### Крок 11. Підсумковий висновок

У новій Markdown-комірці після цієї сформулюйте висновок обсягом 5–8 речень. Обов'язково зазначте:

- яка модель показала найкращий результат;
- за якою метрикою виконано вибір;
- чи є ознаки перенавчання;
- як регуляризація вплинула на результат;
- чому test не використовувався під час підбору гіперпараметрів.

**Підсумковий висновок:**

Найкращий результат показала Lasso з alpha = 0.1: CV RMSE 55.35, на test MAE 42.8, RMSE 53.7, R2 0.456. Модель я обирав за CV RMSE на train, а test використав тільки для фінальної перевірки. Ознак сильного перенавчання немає: train RMSE (53.3-53.7) лише трохи нижчий за CV RMSE (близько 55.4), і на test помилка теж близька. Регуляризація майже не змінила результат: Lasso, ElasticNet і Ridge кращі за звичайну LinearRegression зовсім трохи (CV RMSE 55.35-55.37 проти 55.39), бо ознак мало (10) порівняно з кількістю рядків (442). Зате всі лінійні моделі набагато кращі за DummyRegressor (RMSE 73.2). Test не використовувався під час підбору гіперпараметрів, бо тоді alpha підганялось би під конкретну тестову вибірку і оцінка на test вийшла б занадто оптимістичною. Загалом R2 близько 0.46 означає, що лінійна модель пояснює менше половини розкиду цільової змінної, тому для кращої точності потрібні інші ознаки або нелінійні моделі.

## Контрольні питання після практики

1. Чому `StandardScaler` для Ridge/Lasso/Elastic Net доцільно розміщувати всередині `Pipeline`?
2. Чим відрізняється train-помилка від validation/test-помилки?
3. За якими ознаками можна розпізнати перенавчання?
4. Чим L1-регуляризація відрізняється від L2-регуляризації?
5. Чому `alpha` не можна підбирати за тестовою вибіркою?
6. Яку роль відіграє `l1_ratio` в Elastic Net?
7. Чому великий за модулем коефіцієнт не завжди означає, що ознака є найважливішою?
8. У чому перевага крос-валідації порівняно з одним випадковим validation-розбиттям?

**Відповіді на контрольні питання:**

1. Щоб на кожному кроці CV масштабування рахувалось тільки по train-частині. Якщо масштабувати всі дані заздалегідь, у модель потрапить інформація з validation або test (витік даних).
2. Train-помилка рахується на тих даних, на яких модель вчилась, а validation/test-помилка на нових даних. Тому train-помилка зазвичай менша і сама по собі не показує, як модель працюватиме на нових даних.
3. Train-помилка маленька, а validation/test помітно більша. Ще ознака: при збільшенні складності моделі (наприклад, ступеня полінома) test-помилка починає рости.
4. L1 (Lasso) штрафує суму модулів коефіцієнтів і може зробити частину з них рівними нулю. L2 (Ridge) штрафує суму квадратів і плавно зменшує всі коефіцієнти, але до нуля їх не доводить.
5. Тоді alpha буде підібране під конкретну тестову вибірку, і оцінка на test стане занадто оптимістичною. Test має бути незалежною перевіркою в самому кінці.
6. l1_ratio задає частку L1 у штрафі: 1 означає чистий Lasso, 0 означає чистий Ridge, проміжні значення поєднують обидва.
7. Бо величина коефіцієнта залежить від масштабу ознаки, а ще корельовані ознаки можуть ділити вплив між собою (як s1 і s2 у diabetes).
8. Результат не залежить від одного випадкового поділу: модель перевіряється на кількох різних частинах даних, і видно середню якість та її розкид.

In [ ]:
import pandas as pd
import requests
from io import StringIO

url = 'https://uk.wikipedia.org/wiki/%D0%9D%D0%B0%D1%81%D0%B5%D0%BB%D0%B5%D0%BD%D0%BD%D1%8F_%D0%A3%D0%BA%D1%80%D0%B0%D1%97%D0%BD%D0%B8#%D0%9D%D0%B0%D1%80%D0%BE%D0%B4%D0%B6%D1%83%D0%B2%D0%B0%D0%BD%D1%96%D1%81%D1%82%D1%8C%22'

headers = {
"User-Agent": "Mozilla/5.0",
"Accept-Language": "uk-UA,uk;q=0.9,en;q=0.8",
}

html = requests.get(url, headers=headers, timeout=30).text

df = pd.read_html(
StringIO(html),
match="Коефіцієнт народжуваності в регіонах України",
thousands=".", decimal=","
)[0]

df